# Fairness Over Time — Phase 1: Baseline (Texas, 2014)

**Research question:** Does a model that is fair when deployed stay fair as data changes over time?

This notebook builds the 2014 baseline that the rest of the project compares against:

1. Load and profile the 2014 Texas ACS data
2. Define the sensitive groups (race/ethnicity × sex)
3. Split the data (60/20/20, stratified)
4. Preprocess the features
5. Train logistic regression and XGBoost
6. Measure fairness
7. Mitigate unfairness with Fairlearn (ExponentiatedGradient, equalized odds)
8. Freeze the four models and record the official baseline on the test set

**Rules followed throughout:** all reusable code lives in `src/`; preprocessing is fit on 2014 training data only; every modelling decision was made on the validation set; the test set is used once, at the end.

In [45]:
%load_ext autoreload
%autoreload 2

import time
import warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score

from src.data import DATA_DIR, ARTIFACTS_DIR, load_acs, make_race_eth, split_data, save_split, load_split
from src.preprocess import NUMERIC_COLS, ORDINAL_COLS, NOMINAL_COLS, build_preprocessor
from src.models import make_lr, make_xgb, make_pipeline_for, fit_mitigated, load_models, MODEL_INFO
from src.fairness import (make_sensitive, group_metrics, fairness_summary, fairness_long,
                          bootstrap_gaps, evaluate_all, bootstrap_all, save_results)

warnings.filterwarnings("ignore", module="fairlearn")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load and profile the 2014 data

Data: Folktables **ACSIncome**, 1-Year ACS person survey, **Texas, 2014**. The task is to predict whether a person's total income is over **$50,000**.

In [43]:
print (DATA_DIR)

X, y, meta = load_acs(2014)
print(X.shape, y.shape, meta.shape)

X.head()

/Users/nguyenngochongnhi/Desktop/UTA/Special Topic in AI/Project/fairness-over-time/data/raw
(127039, 10) (127039, 1) (127039, 5)


,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,race_eth
0,53.0,1.0,14.0,1.0,5700.0,4.0,1.0,40.0,2.0,NH White
1,68.0,5.0,22.0,1.0,530.0,48.0,0.0,40.0,1.0,Hispanic
2,64.0,3.0,22.0,1.0,2310.0,48.0,1.0,20.0,2.0,Hispanic
3,54.0,1.0,21.0,1.0,800.0,48.0,0.0,50.0,1.0,NH White
4,51.0,7.0,18.0,1.0,4700.0,39.0,1.0,40.0,2.0,NH White


## Data dictionary: ACSIncome (Texas, 2014)

Each row is one survey respondent. ACSIncome keeps only people aged 17+, with income over $100, who usually worked at least 1 hour per week, and with a survey weight of at least 1.

### Features (`X`)

| Column | Meaning | Type | Codes |
|---|---|---|---|
| `AGEP` | Age in years | Numeric | 17–92 |
| `COW` | Class of worker (who the person works for) | Nominal | 1 private for-profit · 2 private nonprofit · 3 local gov · 4 state gov · 5 federal gov · 6 self-employed, not incorporated · 7 self-employed, incorporated · 8 unpaid family business |
| `SCHL` | Highest education completed | Ordinal | 1 no schooling … 16 HS diploma · 17 GED · 18–19 some college · 20 associate's · 21 bachelor's · 22 master's · 23 professional degree · 24 doctorate |
| `MAR` | Marital status | Nominal | 1 married · 2 widowed · 3 divorced · 4 separated · 5 never married |
| `OCCP` | Occupation | Nominal | 478 codes in 2014 (codes change in 2018) |
| `POBP` | Place of birth | Nominal | 1–56 US states (48 = Texas) · 60–78 US territories · 100+ other countries |
| `RELP` | Relationship to householder | Nominal | 0 householder · 1 spouse · 2+ child, parent, roommate, etc. |
| `WKHP` | Hours usually worked per week | Numeric | 1–99 (99 = 99 or more) |
| `SEX` | Sex (sensitive attribute) | Nominal | 1 male · 2 female |
| `RAC1P` | Race (sensitive attribute) | Nominal | 1 White · 2 Black · 3 American Indian · 4 Alaska Native · 5 AIAN, tribes specified · 6 Asian · 7 Native Hawaiian/Pacific Islander · 8 Some Other Race · 9 Two or More Races |

### Label (`y`)

| Column | Meaning |
|---|---|
| `PINCP` | True if total personal income (all sources) is over $50,000; False otherwise |

### Sensitive group (`group`)

| Column | Meaning |
|---|---|
| `RAC1P` | Copy of the race column, used for fairness analysis |

*Source: 2014 ACS PUMS Data Dictionary (U.S. Census Bureau).*

In [3]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 127039 entries, 0 to 127038
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   AGEP      127039 non-null  float64
 1   COW       127039 non-null  float64
 2   SCHL      127039 non-null  float64
 3   MAR       127039 non-null  float64
 4   OCCP      127039 non-null  float64
 5   POBP      127039 non-null  float64
 6   RELP      127039 non-null  float64
 7   WKHP      127039 non-null  float64
 8   SEX       127039 non-null  float64
 9   race_eth  127039 non-null  str    
dtypes: float64(9), str(1)
memory usage: 10.7 MB


In [4]:
X.describe()

,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX
count,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000,127039.000000
mean,42.690316,2.053275,18.073615,2.415565,4236.300301,86.972977,2.000464,39.755437,1.465715
std,14.778244,1.807224,3.757306,1.749398,2580.129698,100.190467,4.052481,13.325580,0.498825
min,17.000000,1.000000,1.000000,1.000000,10.000000,1.000000,0.000000,1.000000,1.000000
25%,30.000000,1.000000,16.000000,1.000000,2310.000000,47.000000,0.000000,36.000000,1.000000
50%,43.000000,1.000000,19.000000,1.000000,4250.000000,48.000000,0.000000,40.000000,1.000000
75%,54.000000,3.000000,21.000000,5.000000,5700.000000,48.000000,1.000000,45.000000,2.000000
max,92.000000,8.000000,24.000000,5.000000,9830.000000,554.000000,17.000000,99.000000,2.000000


In [5]:
print("Duplicate rows:", X.duplicated().sum())

Duplicate rows: 3239


In [44]:
y.value_counts(normalize=True)

PINCP
False    0.683648
True     0.316352
Name: proportion, dtype: float64

In [7]:
meta['race_eth'].value_counts()

race_eth
NH White    68001
Hispanic    39260
NH Black    12102
NH Asian     5442
NH Other     2234
Name: count, dtype: int64

In [8]:
y["PINCP"].groupby(X["SEX"]).mean()

SEX
1.0    0.393473
2.0    0.227875
Name: PINCP, dtype: float64

### Data profile: key findings

- **No missing values or special codes**; all 10 columns are stored as numbers, even the ones that are really categories (e.g., occupation codes).
- **Positive rate:** 31.6% earn over $50K. A model that always predicts "no" would already be 68.4% accurate, so accuracy alone is a weak measure.
- **Sex gap in the data itself:** 39.3% of men vs. 22.8% of women earn over $50K (16.5 points).
- **Duplicates:** 3,714 rows (2.9%) share identical features with another row. These are different respondents with common profiles, not duplicate records, so they are kept. Dropping them would remove real people unevenly across groups.

In [9]:
import inspect
from folktables import acs
print(inspect.getsource(acs.adult_filter))

def adult_filter(data):
    """Mimic the filters in place for Adult data.

    Adult documentation notes: Extraction was done by Barry Becker from
    the 1994 Census database. A set of reasonably clean records was extracted
    using the following conditions:
    ((AAGE>16) && (AGI>100) && (AFNLWGT>1)&& (HRSWK>0))
    """
    df = data
    df = df[df['AGEP'] > 16]
    df = df[df['PINCP'] > 100]
    df = df[df['WKHP'] > 0]
    df = df[df['PWGTP'] >= 1]
    return df



### Who is in the data

ACSIncome keeps only people who:

| Condition | Meaning | Why |
|---|---|---|
| `AGEP > 16` | Aged 17 or older | Working-age adults |
| `PINCP > 100` | Income over $100 a year | Drops zero or negative incomes |
| `WKHP > 0` | Usually worked at least 1 hour/week | Income prediction is about workers |
| `PWGTP >= 1` | Survey weight of at least 1 | The record represents real people |

This leaves **127,039 people**. Two consequences for interpreting results:

- **Only workers are included.** Comparisons between groups (e.g., women vs. men) are comparisons between *working* women and *working* men.
- **The data is unweighted.** Models and metrics treat every respondent equally, so results describe the survey sample, which approximates but does not exactly match the Texas population.

In [10]:
assert sorted(NUMERIC_COLS + ORDINAL_COLS + NOMINAL_COLS) == sorted(X.columns)

In [11]:
X[NOMINAL_COLS].nunique()

COW           8
MAR           5
OCCP        478
POBP        211
RELP         18
SEX           2
race_eth      5
dtype: int64

## 2. Defining race/ethnicity

ACSIncome's race variable, `RAC1P`, records **race only**. Hispanic origin is a separate Census question (`HISP`). In Texas this matters a lot: the next cells check how Hispanic Texans are spread across the race codes.

Note: in `HISP`, the code **1 means *not* Hispanic**; codes 2–24 are specific Hispanic origins.

In [12]:
from folktables import ACSDataSource, acs
source = ACSDataSource(survey_year="2014", horizon="1-Year", survey="person", root_dir=str(DATA_DIR))
raw = source.get_data(states=["TX"], download=False)

In [13]:
filtered = acs.adult_filter(raw)
len(filtered)

127039

In [14]:
filtered = filtered.reset_index(drop=True)
(filtered["AGEP"].values == X["AGEP"].values).all()

np.True_

In [15]:
filtered.head()

,RT,SERIALNO,SPORDER,PUMA,ST,ADJINC,PWGTP,AGEP,CIT,CITWP,...,pwgtp71,pwgtp72,pwgtp73,pwgtp74,pwgtp75,pwgtp76,pwgtp77,pwgtp78,pwgtp79,pwgtp80
0,P,3,2,3602,48,1008425,82,53,1,NaN,...,81,27,24,21,98,67,69,93,101,21
1,P,5,1,6603,48,1008425,112,68,1,NaN,...,114,99,140,31,110,111,109,178,100,205
2,P,5,2,6603,48,1008425,97,64,1,NaN,...,103,95,100,25,92,90,91,142,101,211
3,P,48,1,3900,48,1008425,35,54,1,NaN,...,42,35,78,79,52,40,10,27,44,45
4,P,48,2,3900,48,1008425,29,51,1,NaN,...,36,30,75,48,36,20,7,35,45,43


In [16]:
pd.crosstab(filtered["RAC1P"], filtered["HISP"] == 1)

HISP,False,True
RAC1P,,
1,31526,68001
2,295,12102
3,141,316
4,0,6
5,150,42
6,68,5442
7,21,75
8,6169,125
9,890,1670


In [17]:
filtered["is_hispanic"] = filtered["HISP"] != 1
y["PINCP"].groupby(filtered["is_hispanic"]).mean()

/var/folders/hh/vzwqsyz92pbdyc1q7kc30_k00000gn/T/ipykernel_26663/3402199326.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  filtered["is_hispanic"] = filtered["HISP"] != 1


is_hispanic
False    0.382187
True     0.169154
Name: PINCP, dtype: float64

### Finding: race alone hides Hispanic Texans

- **30.9%** of the data is Hispanic.
- About **a third of RAC1P "White"** respondents are Hispanic, and **"Some Other Race" is 98% Hispanic**.
- **Positive rate:** 38.2% (non-Hispanic) vs. **16.9% (Hispanic)**, a **21.3-point gap**, larger than the sex gap.

A race-only analysis would miss one of the largest group differences in Texas.

### Decision: a combined race/ethnicity variable (`race_eth`)

| Group | Rule | Count (2014) |
|---|---|---|
| Hispanic | any race, `HISP != 1` | 39,260 |
| NH White | not Hispanic, `RAC1P == 1` | 68,001 |
| NH Black | not Hispanic, `RAC1P == 2` | 12,102 |
| NH Asian | not Hispanic, `RAC1P == 6` | 5,442 |
| NH Other | everyone else (AIAN, NHPI, Some Other Race, Two or More) | 2,234 |

Why this grouping:
- It captures the Hispanic gap that RAC1P misses.
- It solves the small-group problem: several RAC1P groups (e.g., Native Hawaiian/Pacific Islander, 96 people) are too small to evaluate on their own, but every `race_eth` × sex group has roughly 200+ people in the test set.

**Trade-off:** results are not directly comparable with published ACSIncome numbers, which use RAC1P.

`race_eth` is built by `make_race_eth()` in `src/data.py`, and `load_acs()` now returns a `meta` table (race/ethnicity, sex, income in dollars, ADJINC, survey weight) aligned row for row with `X` and `y`.

In [18]:
race_eth = make_race_eth(filtered)
print(race_eth.value_counts())

# One-row test: a White Hispanic person should be "Hispanic"
print(make_race_eth(pd.DataFrame({"RAC1P": [1], "HISP": [2]})))

race_eth
NH White    68001
Hispanic    39260
NH Black    12102
NH Asian     5442
NH Other     2234
Name: count, dtype: int64
0    Hispanic
Name: race_eth, dtype: str


### Handling rare categories

Occupation (`OCCP`) has 478 codes and birthplace (`POBP`) has many too. Categories with fewer than **30 people in the training set** are merged into one "infrequent" column (`OneHotEncoder(min_frequency=30)`):

- Each kept category has at least 30 examples, enough for a stable estimate.
- Only **2.7%** (occupation) and **1.4%** (birthplace) of training rows fall into the infrequent bucket.
- Result: **456 encoded features**.

Codes never seen in 2014 (possible in later years) also go to the infrequent column instead of causing an error. The preprocessor is fit on **2014 training data only** and never refit, so later years are measured with the same 2014 "ruler" and drift stays visible.

In [19]:
X, y, meta = load_acs(2014)
print(X.shape, y.shape, meta.shape)
print(meta.head())
print(meta["race_eth"].value_counts())

(127039, 10) (127039, 1) (127039, 5)
   race_eth  SEX   income   ADJINC  PWGTP
0  NH White    2  36000.0  1008425     82
1  Hispanic    1  82000.0  1008425    112
2  Hispanic    2  55200.0  1008425     97
3  NH White    1  48000.0  1008425     35
4  NH White    2  75000.0  1008425     29
race_eth
NH White    68001
Hispanic    39260
NH Black    12102
NH Asian     5442
NH Other     2234
Name: count, dtype: int64


## 3. Train / validation / test split

- **60 / 20 / 20** split of the 2014 data: 76,223 / 25,408 / 25,408 rows.
- **Stratified by label × race/ethnicity × sex** (20 strata), so every group appears in every set with the same share and the same positive rate. Without this, small groups' true positive rates would vary from split to split by chance.
- **Roles:** train = fit models; validation = all decisions (tuning, mitigation choices); test = used **once**, for the final baseline.
- The split is saved to `artifacts/results/split_2014.csv` so later phases use exactly the same rows.

In [20]:
from src.data import split_data, save_split, load_split
train_idx, val_idx, test_idx = split_data(y, meta)
print(len(train_idx), len(val_idx), len(test_idx))

# Strata check: how many strata, and the smallest?
strata = y["PINCP"].astype(str) + "_" + meta["race_eth"] + "_" + meta["SEX"].astype(str)
print(strata.nunique(), strata.value_counts().min())

# Positive rate per set
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(name, y.loc[idx, "PINCP"].mean())

# Overwrite the saved split
save_split(train_idx, val_idx, test_idx)
tr, va, te = load_split()
print(set(tr) == set(train_idx), set(va) == set(val_idx), set(te) == set(test_idx))

76223 25408 25408
20 216
train 0.3163480839116802
val 0.3163176952141058
test 0.3163964105793451
True True True


In [21]:
print(X.shape)                        # still (127039, 10)
print(X.columns.tolist())             # race_eth present, RAC1P gone
print(X["race_eth"].value_counts())   # same 5 counts as before

(127039, 10)
['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELP', 'WKHP', 'SEX', 'race_eth']
race_eth
NH White    68001
Hispanic    39260
NH Black    12102
NH Asian     5442
NH Other     2234
Name: count, dtype: int64


## 4. Features and preprocessing

### Which features the model sees

The model uses 10 features: age, class of worker, education, marital status, occupation, place of birth, relationship to householder, hours worked, **sex**, and **race/ethnicity** (`race_eth` replaces `RAC1P`).

**Why sex and race/ethnicity are kept as inputs:** removing them ("fairness through unawareness") does not remove bias, because other features carry the same information (occupation, birthplace, household role). Keeping them lets SHAP measure directly how much the model relies on them.

**Caveat:** in US hiring or lending, using protected attributes directly as inputs is generally not allowed. This is a research model for studying how fairness changes over time, not a deployed decision system.

### Column types

| Type | Columns | Treatment |
|---|---|---|
| Numeric | AGEP, WKHP | Standardized |
| Ordinal | SCHL (education codes 1–24, ordered) | Standardized |
| Nominal | COW, MAR, OCCP, POBP, RELP, SEX, race_eth | One-hot encoded |

In [22]:
occ = X["OCCP"].value_counts()
pob = X["POBP"].value_counts()
for t in [10, 30, 50, 100]:
    print(t, "OCCP kept:", (occ >= t).sum(), "| POBP kept:", (pob >= t).sum())

10 OCCP kept: 441 | POBP kept: 162
30 OCCP kept: 362 | POBP kept: 118
50 OCCP kept: 309 | POBP kept: 97
100 OCCP kept: 225 | POBP kept: 72


In [23]:
occ_tr = X.loc[train_idx, "OCCP"].value_counts()
pob_tr = X.loc[train_idx, "POBP"].value_counts()
print("OCCP rows in rare codes:", occ_tr[occ_tr < 30].sum() / occ_tr.sum())
print("POBP rows in rare codes:", pob_tr[pob_tr < 30].sum() / pob_tr.sum())

OCCP rows in rare codes: 0.02744578408092046
POBP rows in rare codes: 0.013906563635648033


In [24]:
pre = build_preprocessor()
Xt_train = pre.fit_transform(X.loc[train_idx])
Xt_val = pre.transform(X.loc[val_idx])
Xt_test = pre.transform(X.loc[test_idx])

print(Xt_train.shape, Xt_val.shape, Xt_test.shape)
names = pre.get_feature_names_out()
print(len(names), [n for n in names if "infrequent" in n])

(76223, 456) (25408, 456) (25408, 456)
456 ['OCCP_infrequent_sklearn', 'POBP_infrequent_sklearn', 'RELP_infrequent_sklearn']


In [25]:
X.loc[train_idx, "RELP"].value_counts().sort_values().head()

RELP
14.0      5
8.0      77
3.0     181
11.0    300
6.0     386
Name: count, dtype: int64

## 5. Models

Two models, each a scikit-learn `Pipeline` (preprocessing + classifier), defined in `src/models.py`:

- **Logistic regression** (simple, interpretable)
- **XGBoost** (captures interactions, usually more accurate)

Both are kept through the whole project rather than picking one "best" model, so we can see whether fairness changes differently for different model types.

In [26]:
y_train = y.loc[train_idx, "PINCP"].astype(int)
y_val = y.loc[val_idx, "PINCP"].astype(int)

for name, model in [("LR", make_lr()), ("XGB", make_xgb())]:
    pipe = make_pipeline_for(model)
    pipe.fit(X.loc[train_idx], y_train)
    proba = pipe.predict_proba(X.loc[val_idx])[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(name, "acc:", accuracy_score(y_val, pred), "AUC:", roc_auc_score(y_val, proba))

LR acc: 0.8230872166246851 AUC: 0.8883523326971859
XGB acc: 0.8282037153652393 AUC: 0.8956414681516556


In [27]:
#tuning
for depth in [4, 6, 8]:
    pipe = make_pipeline_for(make_xgb(max_depth=depth))
    pipe.fit(X.loc[train_idx], y_train)
    proba = pipe.predict_proba(X.loc[val_idx])[:, 1]
    print(depth, roc_auc_score(y_val, proba))

4 0.8922078530541567
6 0.8956414681516556
8 0.8963892688561103


### Validation results and tuning

| Model | Accuracy | AUC |
|---|---|---|
| Always predict "no" | 0.684 | 0.500 |
| Logistic regression | 0.823 | 0.888 |
| XGBoost | 0.828 | 0.896 |

XGBoost tree depth was tuned on validation AUC: depth 4 = 0.8922, **6 = 0.8956**, 8 = 0.8964. Depth 8's gain (< 0.001) is within noise, so the simpler depth 6 was kept. Tuning was kept light on purpose: the project is about fairness over time, not maximum accuracy.

## 6. Measuring fairness

Functions in `src/fairness.py` compute, for each group (sex, race/ethnicity, and their intersection):

| Metric | Meaning |
|---|---|
| Selection rate | Share predicted to earn > $50K |
| True positive rate (TPR) | Of people who **really** earn > $50K, the share the model recognizes |
| False positive rate (FPR) | Of people who really earn ≤ $50K, the share wrongly flagged |

And two summary gaps:
- **Demographic parity difference (DPD):** the largest gap in selection rate between groups
- **Equalized odds difference (EOD):** the largest gap in TPR or FPR between groups

In [28]:
pipe = make_pipeline_for(make_xgb())
pipe.fit(X.loc[train_idx], y_train)
pred_val = pipe.predict(X.loc[val_idx])
sens_val = meta.loc[val_idx]

print(group_metrics(y_val, pred_val, sens_val["SEX"]))
print(group_metrics(y_val, pred_val, sens_val["race_eth"]))
print(group_metrics(y_val, pred_val, sens_val[["race_eth", "SEX"]]))   # intersections

       Count  Selection_rate       TPR       FPR  Accuracy
SEX                                                       
1    13576.0        0.385460  0.747613  0.150577  0.809369
2    11832.0        0.170808  0.545252  0.060311  0.849814
            Count  Selection_rate       TPR       FPR  Accuracy
race_eth                                                       
Hispanic   7852.0        0.101630  0.422005  0.036475  0.872007
NH Asian   1089.0        0.452709  0.830835  0.168810  0.831038
NH Black   2419.0        0.160810  0.523629  0.059259  0.849525
NH Other    447.0        0.241611  0.640000  0.086957  0.836689
NH White  13601.0        0.401882  0.743961  0.163255  0.798618
               Count  Selection_rate       TPR       FPR  Accuracy
race_eth SEX                                                      
Hispanic 1    4347.0        0.136876  0.456803  0.050278  0.844720
         2    3505.0        0.057917  0.341646  0.021263  0.905849
NH Asian 1     581.0        0.559380  0.878472  

In [29]:
for name, sens in [("sex", sens_val["SEX"]),
                   ("race_eth", sens_val["race_eth"]),
                   ("race_eth x sex", sens_val[["race_eth", "SEX"]])]:
    print(name, fairness_summary(y_val, pred_val, sens))

sex {'DPD': np.float64(0.21465165628562197), 'EOD': np.float64(0.20236058107123822)}
race_eth {'DPD': np.float64(0.35107874933281313), 'EOD': np.float64(0.4088305962960035)}
race_eth x sex {'DPD': np.float64(0.5014631176018524), 'EOD': np.float64(0.5368263369354391)}


In [30]:
long = fairness_long(y_val, pred_val, make_sensitive(meta.loc[val_idx]))
print(long.shape)
print(long["attribute"].value_counts())
display(long[long["metric"].isin(["DPD", "EOD"])])

(91, 4)
attribute
race_eth_x_sex    52
race_eth          27
sex               12
Name: count, dtype: int64


,attribute,group,metric,value
10,sex,(gap),DPD,0.214652
11,sex,(gap),EOD,0.202361
37,race_eth,(gap),DPD,0.351079
38,race_eth,(gap),EOD,0.408831
89,race_eth_x_sex,(gap),DPD,0.501463
90,race_eth_x_sex,(gap),EOD,0.536826


### Findings: the unmitigated model (XGBoost, validation)

**The model widens the sex gap.** In the data, men are 16.5 points more likely than women to earn over $50K; the model predicts it 21.4 points more often. Of women who really earn over $50K, the model recognizes only **55%** (men: 75%).

**Hispanic workers are served worst:**

| Group | Selection rate | TPR |
|---|---|---|
| NH Asian | 45.3% | 83.1% |
| NH White | 40.2% | 74.4% |
| NH Other | 24.2% | 64.0% |
| NH Black | 16.1% | 52.4% |
| **Hispanic** | **10.2%** | **42.2%** |

The model **misses 58% of Hispanic Texans who earn over $50K.**

**Accuracy hides this.** Women (85.0%) and Hispanic workers (87.2%) have the *highest* accuracy, because predicting "no" is usually right in groups with fewer high earners. This is why the project tracks TPR, FPR and selection rates, not just accuracy.

Summary gaps: sex EOD 0.20, race/ethnicity EOD 0.41, **race/ethnicity × sex EOD 0.54**. The intersection is largest because it compares the most extreme combinations.

In [31]:
#bootstrap
t = time.time()
ci = bootstrap_gaps(y_val, pred_val, make_sensitive(meta.loc[val_idx]))
print(f"{time.time() - t:.0f} seconds")
display(ci)

140 seconds


,attribute,metric,value,ci_low,ci_high
0,sex,DPD,0.214652,0.202949,0.224089
1,sex,EOD,0.202361,0.181588,0.223165
2,race_eth,DPD,0.351079,0.320976,0.378025
3,race_eth,EOD,0.408831,0.368290,0.449314
4,race_eth_x_sex,DPD,0.501463,0.481724,0.543673
5,race_eth_x_sex,EOD,0.536826,0.476807,0.584277


### How precise are these numbers?

Bootstrap confidence intervals (200 resamples) show how much each gap would vary with a different sample:

| Attribute | EOD | 95% CI |
|---|---|---|
| Sex | 0.202 | [0.182, 0.223] |
| Race/ethnicity | 0.409 | [0.368, 0.449] |
| Race/ethnicity × sex | 0.537 | [0.477, 0.584] |

All intervals are far from 0, so the unfairness is real. The intersection is the noisiest (about ±0.05) because its groups are smallest. **In later years, a change smaller than the interval width is not evidence that fairness changed.**

## 7. Mitigation

**Method:** Fairlearn `ExponentiatedGradient`, which retrains the model many times with re-weighted data until it satisfies a fairness constraint.

**Constraint: equalized odds on race/ethnicity × sex.**
- The main problem found above is the TPR gap: the model misses qualified people in some groups. Equalized odds targets exactly that.
- Demographic parity was rejected: true > $50K rates genuinely differ between groups (16.5 points by sex, 21.3 by Hispanic origin), so forcing equal selection rates would work against real differences in the data, at a larger accuracy cost.
- The constraint uses the intersection because the research question is intersectional.

**Technical notes:**
- Fairlearn passes sample weights that a scikit-learn Pipeline cannot accept, so `fit_mitigated()` fits the preprocessor first, mitigates on the transformed data, then assembles the two fitted parts into one Pipeline.
- The fitted mitigator stores a copy of the training data and every inner model it tried (113 for XGBoost). `MitigatedClassifier` keeps only the inner models with non-zero weight (11 for XGBoost), cutting file sizes from 295 MB / 650 MB to 61 KB / 8.8 MB with identical predictions.
- Fitting takes ~4 min (LR) and ~9 min (XGBoost), so these cells load the saved model if it exists.

In [32]:
# Logistic regression, mitigated: load if already saved, otherwise fit (~4 min) and save
sens_train = make_sensitive(meta.loc[train_idx])[["race_eth", "sex"]]
sens_val = make_sensitive(meta.loc[val_idx])

path = ARTIFACTS_DIR / "models" / "lr_mitigated.joblib"
if path.exists():
    mit_lr = joblib.load(path)
    print("loaded saved model")
else:
    t = time.time()
    mit_lr = fit_mitigated(make_lr(), X.loc[train_idx], y_train, sens_train)
    print(f"fitted in {time.time() - t:.0f} seconds")
    joblib.dump(mit_lr, path)

proba = mit_lr.predict_proba(X.loc[val_idx])[:, 1]
pred = mit_lr.predict(X.loc[val_idx])
print("acc:", accuracy_score(y_val, pred), "AUC:", roc_auc_score(y_val, proba))
print(fairness_summary(y_val, pred, sens_val[["race_eth", "sex"]]))

loaded saved model
acc: 0.8120670654911839 AUC: 0.8068976318703647
{'DPD': np.float64(0.3045283231970065), 'EOD': np.float64(0.20417866570674348)}


In [33]:
lr_pipe = make_pipeline_for(make_lr()).fit(X.loc[train_idx], y_train)
print(fairness_summary(y_val, lr_pipe.predict(X.loc[val_idx]), sens_val[["race_eth", "sex"]]))

{'DPD': np.float64(0.5249515690641106), 'EOD': np.float64(0.5980967719589915)}


In [34]:
print(np.unique(proba).size) 

506


In [35]:
rng = np.random.default_rng(42)
proba = mit_lr.predict_proba(X.loc[val_idx])[:, 1]
pred_rand = (rng.random(len(proba)) < proba).astype(int)   # "yes" with probability = vote share

print("acc:", accuracy_score(y_val, pred_rand))
print(fairness_summary(y_val, pred_rand, sens_val[["race_eth", "sex"]]))

acc: 0.7828636649874056
{'DPD': np.float64(0.19257352484622606), 'EOD': np.float64(0.11032331253545091)}


In [36]:
print(accuracy_score(y_val, pred_rand))

0.7828636649874056


In [37]:
# XGBoost, mitigated: load if already saved, otherwise fit (~9 min) and save
path = ARTIFACTS_DIR / "models" / "xgb_mitigated.joblib"
if path.exists():
    mit_xgb = joblib.load(path)
    print("loaded saved model")
else:
    t = time.time()
    mit_xgb = fit_mitigated(make_xgb(), X.loc[train_idx], y_train, sens_train)
    print(f"fitted in {time.time() - t:.0f} seconds")
    joblib.dump(mit_xgb, path)

proba = mit_xgb.predict_proba(X.loc[val_idx])[:, 1]
pred = mit_xgb.predict(X.loc[val_idx])
print("acc:", accuracy_score(y_val, pred), "AUC:", roc_auc_score(y_val, proba))
print(fairness_summary(y_val, pred, sens_val[["race_eth", "sex"]]))

loaded saved model
acc: 0.8134052267002518 AUC: 0.792919458115851
{'DPD': np.float64(0.2718820666812348), 'EOD': np.float64(0.1840277777777778)}


### Decision: deterministic predictions

Fairlearn's mitigated model is a *mix* of inner models, and its own `predict` is randomized (each person gets "yes" with probability equal to the mix's vote share). The alternative is a fixed rule: "yes" if the vote share is ≥ 0.5. Comparison on validation (logistic regression, race/ethnicity × sex):

| | Accuracy | DPD | EOD |
|---|---|---|---|
| Unmitigated | 0.823 | 0.525 | 0.598 |
| Mitigated, **thresholded at 0.5 (chosen)** | 0.812 | 0.305 | 0.204 |
| Mitigated, randomized | 0.783 | 0.193 | 0.110 |

The thresholded version keeps about two-thirds of the fairness gain for 1.1 points of accuracy; randomizing gets further but costs about 4 points. Fixed decisions are also easier to explain and closer to how real systems decide.

**Limitations:** Fairlearn's fairness guarantee applies to its randomized predictions, so it cannot be claimed for the thresholded version. The mitigated models' AUC is computed from a vote share rather than a calibrated probability, so it is not directly comparable with the unmitigated AUC.

## 8. Freeze the models and record the baseline

The four models are saved to `artifacts/models/` and **never retrained**: `lr`, `lr_mitigated`, `xgb`, `xgb_mitigated`. Every later phase, and the app, loads them with `load_models()`.

The test set is used now, **for the first and only time**. Its results are the official 2014 baseline; no model or setting is changed because of them.

In [38]:
#save the unmitigated pipelines (notebook)
for name, make in [("lr", make_lr), ("xgb", make_xgb)]:
    path = ARTIFACTS_DIR / "models" / f"{name}.joblib"
    if path.exists():
        print(name, "already saved")
        continue
    pipe = make_pipeline_for(make()).fit(X.loc[train_idx], y_train)
    joblib.dump(pipe, path)
    print(name, "fitted and saved")

lr already saved
xgb already saved


In [39]:
models = load_models()
print(list(models))

# Sanity check: validation accuracy should match the Step 6 table exactly
for name, pipe in models.items():
    acc = accuracy_score(y_val, pipe.predict(X.loc[val_idx]))
    print(f"{name:15s} {acc:.3f}")

['lr', 'lr_mitigated', 'xgb', 'xgb_mitigated']
lr              0.823
lr_mitigated    0.812
xgb             0.828
xgb_mitigated   0.813


In [49]:
#evaluate on test set
X_test = X.loc[test_idx]
y_test = y.loc[test_idx, "PINCP"].astype(int)
meta_test = meta.loc[test_idx]

baseline = evaluate_all(models, X_test, y_test, meta_test, year=2014, model_info=MODEL_INFO)
print(baseline.shape)

save_results(baseline, ARTIFACTS_DIR / "results" / "metrics_over_time.csv")

# Headline view: overall performance + intersectional gaps
headline = baseline[
    (baseline["attribute"].isin(["overall", "race_eth_x_sex"]))
    & (baseline["metric"].isin(["accuracy", "auc", "DPD", "EOD"]))
]
display(headline.pivot_table(index=["model", "mitigated"], columns="metric", values="value"))

(372, 7)


metric                  DPD       EOD  accuracy       auc
model   mitigated                                        
LR      False      0.486683  0.523839  0.821080  0.887716
        True       0.282621  0.151351  0.810020  0.806207
XGBoost False      0.502122  0.468679  0.826747  0.895916
        True       0.250823  0.163645  0.813563  0.795656

In [41]:
t = time.time()
ci_2014 = bootstrap_all(models, X_test, y_test, meta_test, year=2014, model_info=MODEL_INFO)
print(f"{time.time() - t:.0f} seconds")

display(ci_2014[ci_2014["attribute"] == "race_eth_x_sex"])

558 seconds


,year,model,mitigated,attribute,metric,value,ci_low,ci_high
4,2014,LR,False,race_eth_x_sex,DPD,0.486683,0.473417,0.506065
5,2014,LR,False,race_eth_x_sex,EOD,0.523839,0.473341,0.587665
10,2014,LR,True,race_eth_x_sex,DPD,0.282621,0.272004,0.319922
11,2014,LR,True,race_eth_x_sex,EOD,0.151351,0.115937,0.280989
16,2014,XGBoost,False,race_eth_x_sex,DPD,0.502122,0.487261,0.515507
17,2014,XGBoost,False,race_eth_x_sex,EOD,0.468679,0.421367,0.545724
22,2014,XGBoost,True,race_eth_x_sex,DPD,0.250823,0.238753,0.274721
23,2014,XGBoost,True,race_eth_x_sex,EOD,0.163645,0.116533,0.280291


In [47]:
save_results(ci_2014, ARTIFACTS_DIR / "results" / "ci_over_time.csv")

,year,model,mitigated,attribute,metric,value,ci_low,ci_high
0,2014,LR,False,race_eth,DPD,0.291592,0.276355,0.319735
1,2014,LR,False,race_eth,EOD,0.302857,0.263006,0.361330
2,2014,LR,False,race_eth_x_sex,DPD,0.486683,0.473417,0.506065
3,2014,LR,False,race_eth_x_sex,EOD,0.523839,0.473341,0.587665
4,2014,LR,False,sex,DPD,0.220267,0.210224,0.231332
5,2014,LR,False,sex,EOD,0.225033,0.207142,0.247762
6,2014,LR,True,race_eth,DPD,0.182972,0.171745,0.215107
7,2014,LR,True,race_eth,EOD,0.099822,0.067069,0.179894
8,2014,LR,True,race_eth_x_sex,DPD,0.282621,0.272004,0.319922
9,2014,LR,True,race_eth_x_sex,EOD,0.151351,0.115937,0.280989


### Official 2014 baseline (test set, race/ethnicity × sex)

| Model | Accuracy | AUC | DPD | EOD [95% CI] |
|---|---|---|---|---|
| LR | 0.821 | 0.888 | 0.487 | 0.524 [0.473, 0.588] |
| LR mitigated | 0.810 | 0.806* | 0.283 | 0.151 [0.116, 0.281] |
| XGBoost | 0.827 | 0.896 | 0.502 | 0.469 [0.421, 0.546] |
| XGBoost mitigated | 0.814 | 0.796* | 0.251 | 0.164 [0.117, 0.280] |

\* From a vote share; not directly comparable with the unmitigated AUC.

**What it shows:**
- **Mitigation works:** intersectional EOD falls by 71% (LR) and 65% (XGBoost) for about 1 point of accuracy. The unmitigated and mitigated intervals do not overlap, so the effect is real.
- **After mitigation, XGBoost's accuracy advantage disappears.**
- **The test numbers match validation** for accuracy and AUC; EOD differs by 0.02–0.07, within the noise level.
- **The mitigated intervals are wide and lopsided** (≈ 0.12–0.28). EOD is the *largest* of many group gaps, and resampling noise pushes that maximum upward when the true gaps are small. In Phase 2, changes in a mitigated model's EOD smaller than about 0.1 should not be read as real change.

Results are saved to `artifacts/results/metrics_over_time.csv` and `ci_over_time.csv`; Phase 2 adds 2015–2017 to the same files.

## Summary and next steps

**Phase 1 established:**
- In 2014 Texas data, an income model trained without constraints is markedly unfair, especially at the intersection of race/ethnicity and sex (EOD ≈ 0.5), and the largest single gap affects **Hispanic workers**.
- Equalized-odds mitigation cuts that gap by about two-thirds for about 1 point of accuracy.

**Next, Phase 2:** run these four frozen models, unchanged, on 2015, 2016 and 2017 data, and ask whether the gaps stay where they were in 2014.

**Limitations to keep in mind:** one state; unweighted survey data; workers only; a fixed $50K threshold while incomes and prices change over time (examined in Phase 3).

In [50]:
print((ARTIFACTS_DIR / "results" / "metrics_over_time.csv").resolve())

/Users/nguyenngochongnhi/Desktop/UTA/Special Topic in AI/Project/fairness-over-time/artifacts/results/metrics_over_time.csv
